# Fine-Tuning Data Size and Epistemic Diversity — Experiment Notebook

**Research question.** How does the size of instruction-tuning data affect the epistemic diversity of a small open-source base LLM?

- **H1 (collapse view, Kirk et al., 2024):** diversity *decreases* as fine-tuning data grows from 1k → 10k.
- **H2 (calibration view, Skobelev et al., 2026):** diversity moves *toward that of the fine-tuning data* (up if the base model starts below Dolly, down otherwise).

| Stage | What it does | Hardware |
|---|---|---|
| 0. Setup & config | Installs, all experiment settings in one cell | — |
| 1. Data | Random Dolly-15k subsets (1k / 5k / 10k × 3 seeds) | CPU |
| 2. Fine-tuning | QLoRA on Qwen3-8B-Base with TRL `SFTTrainer`, **fixed optimizer steps** across sizes | 1 GPU ≥ 24 GB |
| 3. Generation | IssueBench templates × topics from Wright et al.'s `llm-knowledge` repo, identical prompts & decoding for all 10 models | 1 GPU ≥ 24 GB |
| 4a. Claim decomposition | Llama-3.1-70B-Instruct with the paper's prompt (`llm_knowledge.extract_claims_bulk`) | 70B OpenAI-compatible endpoint (vLLM server or API) |
| 4b. Clustering | Mutual-entailment meaning classes (`cluster_entailment` + `break_up_clusters`), pooled per topic across all models | 1 small GPU |
| 4c. Diversity | Hill–Shannon diversity, rarefied to the minimum sample coverage per topic (`llm-knowledge`), exact interpolation as a check | CPU |
| 5. Analysis | Linear mixed-effects models, figures, H1/H2 read-out | CPU |
| 6. Diagnostics | Loss curves, confounds, manual-validation samples | CPU |

**How to use.** Set `DRY_RUN = True` first: it runs everything end-to-end with Qwen3-0.6B-Base, tiny subsets, and 2 topics in a few minutes. Then set it to `False` for the real experiment. Every stage caches its outputs to `PROJECT_DIR` and skips work already done, so you can stop, restart the runtime, and continue.

**Files.** The notebook is self-contained: helper functions are in the first code cell after the installs. Stages 4a–4c use Wright et al.'s package [`llm-knowledge`](https://github.com/dwright37/llm-knowledge); importing it needs a **GPU runtime**.

## 0. Setup

In [ ]:
# Qwen3 needs transformers >= 4.51
!pip -q install -U "transformers>=4.51" "peft>=0.13" "trl>=0.20" "bitsandbytes>=0.43" accelerate datasets \
    "llm-knowledge==0.1.3" openai statsmodels scipy pandas pyarrow matplotlib tqdm
!python -m spacy download -q en_core_web_sm   # required by llm-knowledge
# Optional, for fast generation (Stage 3, GEN_BACKEND = "vllm"):
# !pip -q install vllm

In [ ]:
# =========================== HELPER FUNCTIONS ===========================
# File I/O, prompt format, condition names, the Llama decomposition API wrapper,
# and exact coverage/diversity functions (robustness check in Stage 4c).
import gc, json, random, re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from scipy.special import gammaln

# ---------------------------------------------------------------------------
# One prompt format for training, generation, and the untuned base model
# ---------------------------------------------------------------------------
PROMPT_NO_CTX = "### Instruction:\n{instruction}\n\n### Response:\n"
PROMPT_CTX = "### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
# The base model never learned to emit EOS, so all models are cut at the next turn marker.
STOP_STRINGS = ["### Instruction:", "### Response:", "### Context:"]


def format_prompt(instruction, context=""):
    context = (context or "").strip()
    if context:
        return PROMPT_CTX.format(instruction=instruction.strip(), context=context)
    return PROMPT_NO_CTX.format(instruction=instruction.strip())


def cut_at_stop(text):
    hits = [text.find(s) for s in STOP_STRINGS if s in text]
    return (text[:min(hits)] if hits else text).strip()


# ---------------------------------------------------------------------------
# Small utilities
# ---------------------------------------------------------------------------
def set_all_seeds(s):
    import torch
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def free_gpu():
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def write_jsonl(path, rows, mode="w"):
    with open(path, mode, encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


# ---------------------------------------------------------------------------
# Condition names: "base" or "n<size>_s<seed>"
# ---------------------------------------------------------------------------
def cond_name(size, seed):
    return f"n{size}_s{seed}"


def size_label(n):
    return "base" if n == 0 else (f"{n // 1000}k" if n >= 1000 else str(n))


def parse_cond(c):
    if c == "base":
        return 0, "base"
    n, s = re.match(r"n(\d+)_s(\d+)", c).groups()
    return int(n), s


# ---------------------------------------------------------------------------
# Claim decomposition model (any OpenAI-compatible endpoint)
# ---------------------------------------------------------------------------
class APIModel:
    """Stand-in for llm_knowledge's model object so `extract_claims_bulk` can call
    any OpenAI-compatible endpoint (a vLLM server or a hosted provider).

    `tokenizer = None` makes llm_knowledge chunk responses by sentences only.
    Failed calls return "" (no claims) and are counted in `self.failures`.
    """

    def __init__(self, model_id, base_url, api_key, workers=16):
        from openai import OpenAI
        self.model_id, self.tokenizer, self.workers = model_id, None, workers
        self.client = OpenAI(base_url=base_url, api_key=api_key, max_retries=6, timeout=300)
        self.failures = 0

    def _one(self, msgs, max_new_tokens, temperature, top_p):
        try:
            r = self.client.chat.completions.create(
                model=self.model_id, messages=msgs, max_tokens=max_new_tokens,
                temperature=temperature, top_p=top_p)
            return r.choices[0].message.content or ""
        except Exception as e:
            self.failures += 1
            print("decomposition call failed:", type(e).__name__, str(e)[:150])
            return ""

    def generate(self, messages, max_new_tokens=700, do_sample=True, top_p=0.9,
                 temperature=1.0, **kwargs):
        from tqdm.auto import tqdm
        temperature = temperature if do_sample else 0.0
        with ThreadPoolExecutor(self.workers) as ex:
            texts = list(tqdm(ex.map(lambda m: self._one(m, max_new_tokens, temperature, top_p), messages),
                              total=len(messages), leave=False))
        return {"text": texts}


# ---------------------------------------------------------------------------
# Exact coverage-based rarefaction (Chao & Jost 2012; Chao et al. 2014, as in iNEXT)
# Used as a deterministic robustness check next to llm_knowledge's resampling.
# ---------------------------------------------------------------------------
def _lc(n, k):
    """log C(n, k), -inf where k > n."""
    n, k = np.asarray(n, float), np.asarray(k, float)
    with np.errstate(invalid="ignore"):
        out = gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)
    return np.where((k >= 0) & (k <= n), out, -np.inf)


def coverage_observed(x):
    x = np.asarray(x); x = x[x > 0]; n = x.sum()
    f1, f2 = (x == 1).sum(), (x == 2).sum()
    if n == 0:
        return np.nan
    if f1 == 0:
        return 1.0
    a = (n - 1) * f1 / ((n - 1) * f1 + 2 * f2) if f2 > 0 else (n - 1) * (f1 - 1) / ((n - 1) * (f1 - 1) + 2)
    return float(1 - f1 / n * a)


def coverage_at(x, m):
    x = np.asarray(x); x = x[x > 0]; n = x.sum()
    if m >= n:
        return coverage_observed(x)
    return float(1 - np.sum(x / n * np.exp(_lc(n - x, m) - _lc(n - 1, m))))


def hill_shannon_at(x, m):
    x = np.asarray(x); x = x[x > 0]; n = int(x.sum()); m = int(m)
    if m >= n:
        p = x / n
        return float(np.exp(-np.sum(p * np.log(p))))
    vals, mult = np.unique(x, return_counts=True)
    fk = np.zeros(m + 1)
    for xi, c in zip(vals, mult):
        k = np.arange(1, min(xi, m) + 1)
        fk[k] += c * np.exp(_lc(xi, k) + _lc(n - xi, m - k) - _lc(n, m))
    k = np.arange(1, m + 1)
    return float(np.exp(-np.sum(k / m * np.log(k / m) * fk[1:])))


def size_for_coverage(x, target):
    """Smallest sample size m whose interpolated coverage reaches the target (m = n if needed)."""
    n = int(np.sum(x))
    if n <= 2 or coverage_at(x, n - 1) < target:
        return n
    lo, hi = 1, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if coverage_at(x, mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


def exact_rarefied_hsd(x, target):
    """Hill-Shannon diversity of abundance vector x, interpolated to coverage `target`."""
    return hill_shannon_at(x, size_for_coverage(x, target))

In [ ]:
# =========================== EXPERIMENT CONFIG ===========================
import os, json, math, random, re, gc, time
from pathlib import Path
import numpy as np

DRY_RUN = True   # True = tiny end-to-end smoke test (small model, few steps, 2 topics)

PROJECT_DIR = Path("./epistemic_diversity")
# On Colab, keep outputs on Drive so they survive disconnects:
# from google.colab import drive; drive.mount("/content/drive")
# PROJECT_DIR = Path("/content/drive/MyDrive/epistemic_diversity")

# ---- Model & fine-tuning (plan: "Model", "Fine-tuning") ----
BASE_MODEL       = "Qwen/Qwen3-8B-Base"      # pretrained only, no instruction tuning
DATA_SIZES       = [1_000, 5_000, 10_000]
SEEDS            = [0, 1, 2]                 # [3] sampling seeds per size
PER_DEVICE_BATCH = 4
GRAD_ACCUM       = 4
EFFECTIVE_BATCH  = PER_DEVICE_BATCH * GRAD_ACCUM
# [N] fixed optimizer steps for ALL sizes. Default = one epoch of the largest subset,
# so 10k sees 1 epoch, 5k sees 2, 1k sees 10 (smaller subsets are repeated).
N_STEPS          = math.ceil(max(DATA_SIZES) / EFFECTIVE_BATCH)
LEARNING_RATE    = 2e-4
LR_SCHEDULER     = "cosine"
WARMUP_STEPS     = max(1, int(0.03 * N_STEPS))
MAX_SEQ_LEN      = 1024
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS     = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- Generation (plan: "Generation") ----
# Topics and IssueBench templates come from Wright et al.'s llm-knowledge repo, pinned to one commit.
LLM_KNOWLEDGE_DATA = ("https://raw.githubusercontent.com/dwright37/llm-knowledge/"
                      "b30ec074ce27e2c909a924a540f1762c94170ebd/experiments/data")
# TODO: replace with the [10] general topics you pick from Wright et al.'s topics.txt (Stage 3 prints all 155).
# Set TOPICS = None to sample N_TOPICS at random instead (their list is mostly people and events).
TOPICS = [
    "climate change", "artificial intelligence", "vaccination", "immigration",
    "nuclear energy", "social media", "remote work", "genetically modified food",
    "universal basic income", "space exploration",
]
N_TOPICS     = 10        # only used when TOPICS = None
TOPIC_LIMIT  = None      # use only the first k topics (dry run)
TEMPLATES_PER_TOPIC = 50                     # [50]
TEMPERATURE, TOP_P, MAX_NEW_TOKENS = 1.0, 0.9, 512
GEN_SEED    = 1234
GEN_BACKEND = "vllm"   # "vllm": fast, bf16 base + LoRA | "hf": slower, 4-bit base exactly as in training
HF_GEN_BATCH = 16

# ---- Measurement (plan: "Measurement", following Wright et al., 2026) ----
DECOMP_MODEL    = "meta-llama/Llama-3.1-70B-Instruct"
# Any OpenAI-compatible endpoint: a vLLM server you start, or a hosted provider.
#   e.g. on a 2-4 GPU node:  vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 4
DECOMP_BASE_URL = os.environ.get("DECOMP_BASE_URL", "http://localhost:8000/v1")
DECOMP_API_KEY  = os.environ.get("DECOMP_API_KEY", "EMPTY")
DECOMP_WORKERS  = 16
NLI_MODEL       = "microsoft/deberta-large-mnli"
NLI_TOP_N       = 6        # nearest-neighbour claims checked with NLI per claim (llm-knowledge README: 6)
NLI_CHECKPOINT  = 50_000   # claims between checkpoint saves during clustering (must be > 0)
N_RESAMPLES     = 10       # random rarefaction draws averaged per model x topic
DIVERSITY_RESPONSE = "hsd"  # or "log_hsd" / "hsd_exact" for the mixed models

if DRY_RUN:
    BASE_MODEL = "Qwen/Qwen3-0.6B-Base"
    DATA_SIZES, SEEDS = [50, 100, 200], [0, 1]
    N_STEPS, WARMUP_STEPS = 10, 1
    TOPIC_LIMIT, TEMPLATES_PER_TOPIC, MAX_NEW_TOKENS = 2, 5, 128
    N_RESAMPLES = 3
    PROJECT_DIR = PROJECT_DIR / "dry_run"
    # DECOMP_MODEL = "meta-llama/Llama-3.1-8B-Instruct"   # cheaper for smoke tests only

# ---- Paths & condition names ----
DATA_DIR, ADAPTER_DIR, GEN_DIR, CLAIM_DIR, CLUSTER_DIR, RESULT_DIR, FIG_DIR = [
    PROJECT_DIR / d for d in ["data", "adapters", "generations", "claims", "clusters", "results", "figures"]]
for d in [DATA_DIR, ADAPTER_DIR, GEN_DIR, CLAIM_DIR, CLUSTER_DIR, RESULT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONDITIONS = ["base"] + [cond_name(n, s) for n in DATA_SIZES for s in SEEDS]

config_snapshot = {k: v for k, v in globals().items()
                   if k.isupper() and isinstance(v, (int, float, str, list, bool))}
(PROJECT_DIR / "config.json").write_text(json.dumps(config_snapshot, indent=2, default=str))

print(f"DRY_RUN={DRY_RUN} | {len(CONDITIONS)} models | {TOPIC_LIMIT or N_TOPICS} topics x {TEMPLATES_PER_TOPIC} templates")
print(f"Fixed steps N={N_STEPS}, effective batch={EFFECTIVE_BATCH}")
for n in DATA_SIZES:
    print(f"  {size_label(n):>4}: {N_STEPS * EFFECTIVE_BATCH / n:.1f} epochs")

In [ ]:
import torch, pandas as pd
from tqdm.auto import tqdm

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Data: random Dolly-15k subsets

For each seed we draw one random permutation of Dolly and take its first 1k / 5k / 10k examples, so within a seed the subsets are **nested** (1k ⊂ 5k ⊂ 10k). This keeps the comparison between sizes about *how much* data, not *which* data. Different seeds give independent draws.

In [ ]:
from datasets import load_dataset

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(dolly)

subset_path = DATA_DIR / "subsets.json"
SUBSETS = {}
for seed in SEEDS:
    perm = np.random.default_rng(seed).permutation(len(dolly))
    for n in DATA_SIZES:
        SUBSETS[cond_name(n, seed)] = perm[:n].tolist()
subset_path.write_text(json.dumps(SUBSETS))

# Category mix per subset (should be close to the full dataset's)
cat = pd.Series(dolly["category"])
mix = pd.DataFrame({"full": cat.value_counts(normalize=True)} |
                   {c: cat.iloc[idx].value_counts(normalize=True) for c, idx in SUBSETS.items()}).round(3)
mix

## 2. Fine-tuning (QLoRA, fixed number of optimizer steps)

All 9 runs use the same `N_STEPS`, batch size, learning-rate schedule, and LoRA settings, so **only the amount of distinct data differs**. The trainer cycles through smaller subsets for more epochs.

TRL's `SFTTrainer` does the tokenization, padding, and LoRA wrapping. Each example is given as a `prompt` / `completion` pair, so the loss is computed on response tokens only, and TRL appends an EOS token to each response so the model learns to stop (checked below). Very long Dolly contexts are cut to `MAX_SEQ_LEN // 2` tokens so the response always fits.

Each finished adapter is saved to `adapters/<condition>/`; re-running skips finished runs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("eos:", repr(tok.eos_token), "| pad:", repr(tok.pad_token))

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)

def load_4bit_base():   # also used by GEN_BACKEND = "hf" in Stage 3
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=BNB,
                                                device_map={"": 0}, torch_dtype=torch.bfloat16)

MAX_CTX_TOKENS = MAX_SEQ_LEN // 2

def to_prompt_completion(ex):
    ctx_ids = tok(ex["context"] or "", add_special_tokens=False)["input_ids"]
    ctx = tok.decode(ctx_ids[:MAX_CTX_TOKENS]) if len(ctx_ids) > MAX_CTX_TOKENS else ex["context"]
    return {"prompt": format_prompt(ex["instruction"], ctx), "completion": ex["response"].strip()}

def train_one(size, seed):
    cond = cond_name(size, seed)
    out = ADAPTER_DIR / cond
    if (out / "adapter_config.json").exists():
        print(f"[skip] {cond} already trained"); return
    set_all_seeds(seed)
    ds = dolly.select(SUBSETS[cond]).map(to_prompt_completion, remove_columns=dolly.column_names)
    trainer = SFTTrainer(
        model=load_4bit_base(),
        processing_class=tok,
        train_dataset=ds,
        peft_config=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                               target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"),
        args=SFTConfig(
            output_dir=str(PROJECT_DIR / "tmp_trainer" / cond),
            max_steps=N_STEPS, per_device_train_batch_size=PER_DEVICE_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
            lr_scheduler_type=LR_SCHEDULER, warmup_steps=WARMUP_STEPS,
            max_length=MAX_SEQ_LEN, completion_only_loss=True,
            optim="paged_adamw_8bit", bf16=True,
            gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
            logging_steps=10, save_strategy="no", seed=seed, data_seed=seed, report_to="none",
        ),
    )
    # Check that TRL appended EOS, so the fine-tuned models learn to stop
    ends_eos = np.mean([ids[-1] == tok.eos_token_id for ids in trainer.train_dataset[:200]["input_ids"]])
    assert ends_eos > 0.9, f"only {ends_eos:.0%} of training examples end with EOS"
    t0 = time.time(); trainer.train()
    trainer.save_model(out)
    meta = {"cond": cond, "size": size, "seed": seed, "steps": N_STEPS,
            "epochs": N_STEPS * EFFECTIVE_BATCH / size, "minutes": (time.time() - t0) / 60,
            "log_history": trainer.state.log_history}
    (out / "train_meta.json").write_text(json.dumps(meta, indent=2))
    print(f"[done] {cond}: {meta['epochs']:.1f} epochs in {meta['minutes']:.1f} min")
    del trainer; free_gpu()

In [ ]:
for n in DATA_SIZES:
    for s in SEEDS:
        train_one(n, s)

## 3. Generation

Prompts are IssueBench writing-assistance templates (Röttger et al., 2025) filled with each topic. Both the templates (placeholder `{proposition}`) and the topic list come from Wright et al.'s `llm-knowledge` repository, so the prompts match their setup. The same `TEMPLATES_PER_TOPIC` templates are used for every topic, and **all 10 models (untuned base + 9 adapters) get identical prompts and decoding settings**. Sampling settings the model's own `generation_config` might add (top-k, repetition penalty) are switched off explicitly so only temperature and top-p apply.

In [ ]:
import urllib.request
fetch = lambda name: urllib.request.urlopen(f"{LLM_KNOWLEDGE_DATA}/{name}").read().decode("utf-8")

TEMPLATES  = json.loads(fetch("issuebench_templates.json"))["templates"]
ALL_TOPICS = [t.strip() for t in fetch("topics.txt").splitlines() if t.strip()]

if TOPICS is None:
    TOPICS = [ALL_TOPICS[i] for i in np.random.default_rng(GEN_SEED).permutation(len(ALL_TOPICS))[:N_TOPICS]]
TOPICS = TOPICS[:TOPIC_LIMIT]
chosen = np.random.default_rng(GEN_SEED + 1).permutation(len(TEMPLATES))[:TEMPLATES_PER_TOPIC]
print(f"{len(ALL_TOPICS)} topics in Wright et al.'s list:", ALL_TOPICS)
print(f"\n{len(TEMPLATES)} templates available. Topics used:", TOPICS)

PROMPTS = []
for topic in TOPICS:
    for t in chosen:
        instr = TEMPLATES[t].replace("{proposition}", topic).strip()
        PROMPTS.append({"prompt_id": f"{topic}|{t}", "topic": topic, "template_id": int(t),
                        "instruction": instr, "prompt": format_prompt(instr)})
write_jsonl(DATA_DIR / "prompts.jsonl", PROMPTS)
print(len(PROMPTS), "prompts. Example:\n" + PROMPTS[0]["prompt"])

In [ ]:
def save_generations(cond, texts_ntok):
    size, seed = parse_cond(cond)
    rows = [{"cond": cond, "size": size, "seed": seed, **{k: p[k] for k in ["prompt_id", "topic", "template_id"]},
             "raw": raw, "response": cut_at_stop(raw), "n_new_tokens": ntok}
            for p, (raw, ntok) in zip(PROMPTS, texts_ntok)]
    write_jsonl(GEN_DIR / f"{cond}.jsonl", rows)

def is_done(cond):
    return len(read_jsonl(GEN_DIR / f"{cond}.jsonl")) == len(PROMPTS)

todo = [c for c in CONDITIONS if not is_done(c)]
print("To generate:", todo)

if todo and GEN_BACKEND == "vllm":
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    llm = LLM(model=BASE_MODEL, dtype="bfloat16", enable_lora=True, max_lora_rank=LORA_R,
              max_model_len=MAX_SEQ_LEN + MAX_NEW_TOKENS, seed=GEN_SEED, gpu_memory_utilization=0.85)
    sp = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=-1, min_p=0.0,
                        repetition_penalty=1.0, max_tokens=MAX_NEW_TOKENS, stop=STOP_STRINGS)
    texts = [p["prompt"] for p in PROMPTS]
    for i, cond in enumerate(todo):
        lora = None if cond == "base" else LoRARequest(cond, i + 1, str(ADAPTER_DIR / cond))
        outs = llm.generate(texts, sp, lora_request=lora)
        save_generations(cond, [(o.outputs[0].text, len(o.outputs[0].token_ids)) for o in outs])
        print("[done]", cond)
    del llm; free_gpu()

elif todo and GEN_BACKEND == "hf":
    from peft import PeftModel
    tok.padding_side = "left"
    for cond in todo:
        set_all_seeds(GEN_SEED)
        model = load_4bit_base()
        if cond != "base":
            model = PeftModel.from_pretrained(model, str(ADAPTER_DIR / cond))
        model.eval()
        results = []
        for i in tqdm(range(0, len(PROMPTS), HF_GEN_BATCH), desc=cond):
            batch = [p["prompt"] for p in PROMPTS[i:i + HF_GEN_BATCH]]
            enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                g = model.generate(**enc, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, top_k=0,
                                   repetition_penalty=1.0, max_new_tokens=MAX_NEW_TOKENS,
                                   pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
                                   stop_strings=STOP_STRINGS, tokenizer=tok)
            new = g[:, enc["input_ids"].shape[1]:]
            for seq in new:
                results.append((tok.decode(seq, skip_special_tokens=True), int((seq != tok.pad_token_id).sum())))
        save_generations(cond, results)
        del model; free_gpu()
    tok.padding_side = "right"

In [ ]:
gen = pd.concat([pd.DataFrame(read_jsonl(GEN_DIR / f"{c}.jsonl")) for c in CONDITIONS], ignore_index=True)
gen["n_words"] = gen.response.str.split().str.len()
gen["empty"] = gen.n_words < 3
gen["hit_max_tokens"] = gen.n_new_tokens >= MAX_NEW_TOKENS
gen.groupby("cond", sort=False)[["n_words", "empty", "hit_max_tokens"]].mean().round(3)

In [ ]:
# Eyeball one prompt across models
pid = PROMPTS[0]["prompt_id"]
for c in CONDITIONS[:4]:
    r = gen[(gen.cond == c) & (gen.prompt_id == pid)].response.iloc[0]
    print(f"===== {c} =====\n{r[:800]}\n")

## 4a. Claim decomposition (Llama-3.1-70B-Instruct)

Each response is split into sentence chunks and every chunk into atomic claims, using **Wright et al.'s own decomposition prompt and code** (`llm_knowledge.extract_claims_bulk`, which samples at temperature 1.0 as in the paper). The 70B model is called through any OpenAI-compatible endpoint (`DECOMP_BASE_URL`): a vLLM server you start, or a hosted provider. Results are saved per model to `claims/<condition>.parquet`, so an interrupted run resumes at the next model.

> Keep the 70B model for the real results; a smaller model is only for smoke tests.

In [ ]:
# Importing llm_knowledge needs a GPU runtime (it calls torch.cuda at import time)
from llm_knowledge.epistemic_diversity import extract_claims_bulk, FACTOID_EXTRACTION_PROMPT

decomp_model = APIModel(DECOMP_MODEL, DECOMP_BASE_URL, DECOMP_API_KEY, workers=DECOMP_WORKERS)

# Quick connectivity test with the paper's prompt
test = FACTOID_EXTRACTION_PROMPT.replace("{issue}", "coffee").replace(
    "{content}", "Coffee contains caffeine. Many people drink it in the morning because it helps them feel alert.")
print(decomp_model.generate([[{"role": "user", "content": test}]])["text"][0])

In [ ]:
def clean_claims(f):
    f = f.assign(factoid=f.factoid.str.strip().str.lstrip("-*• ").str.strip())
    return f[(f.factoid.str.len() > 0) & (f.factoid.str.upper().str.rstrip(".") != "EMPTY")]

for cond in CONDITIONS:
    out = CLAIM_DIR / f"{cond}.parquet"
    if out.exists(): print(f"[skip] {cond}"); continue
    g = gen[gen.cond == cond].reset_index(drop=True).assign(text=lambda d: d.response, model_id=cond)
    decomp_model.failures = 0
    f = extract_claims_bulk(g[["text", "topic", "model_id"]], decomp_model, group_key="topic")
    f["prompt_id"] = g.prompt_id.to_numpy()[f.original_index.to_numpy(dtype=int)]
    clean_claims(f).to_parquet(out, index=False)
    print(f"[done] {cond}: {len(f)} claims" +
          (f"  WARNING: {decomp_model.failures} API calls failed (no claims kept for them); "
           f"delete {out.name} and re-run to retry" if decomp_model.failures else ""))

# One row per claim, and one row per response (responses without claims count as 0)
claims = (pd.concat([pd.read_parquet(CLAIM_DIR / f"{c}.parquet") for c in CONDITIONS], ignore_index=True)
            .rename(columns={"model_id": "cond", "factoid": "claim"}))
per_resp = claims.groupby(["cond", "prompt_id"]).claim.apply(list).rename("claims").reset_index()
cl = gen[["cond", "prompt_id", "topic"]].merge(per_resp, on=["cond", "prompt_id"], how="left")
cl["claims"] = cl.claims.apply(lambda x: x if isinstance(x, list) else [])
cl["n_claims"] = cl.claims.str.len()
cl.groupby("cond", sort=False).n_claims.agg(["mean", "median", lambda s: (s == 0).mean()]).round(2)

## 4b. Meaning classes by mutual entailment

All claims for one topic, **pooled across all 10 models**, are grouped so that every model is counted against the same set of meaning classes. This uses Wright et al.'s own algorithm from `llm-knowledge`:

1. `cluster_entailment`: each claim is compared with its `NLI_TOP_N` most similar claims (sentence embeddings only shortlist candidates); it joins a class when the NLI model (`microsoft/deberta-large-mnli`) says the two claims **entail each other in both directions**.
2. `break_up_clusters`: very large classes (≥ 100 claims) are split with DBSCAN on embeddings, and the leftover claims are re-matched with NLI.

Claims still unassigned after step 2 become their own (singleton) meaning class. Each class is labelled with its most frequent claim (`class_rep`) for inspection.

In [ ]:
from transformers import pipeline
from llm_knowledge.epistemic_diversity import cluster_entailment, break_up_clusters

on_gpu = torch.cuda.is_available()
nli = pipeline("text-classification", model=NLI_MODEL, device=0 if on_gpu else -1,
               torch_dtype=torch.float16 if on_gpu else None)

for topic in TOPICS:
    out = CLUSTER_DIR / f"{slug(topic)}.parquet"
    if out.exists(): print(f"[skip] {topic}"); continue
    g = (claims[claims.topic == topic].reset_index(drop=True)
           .rename(columns={"claim": "factoid"})[["cond", "prompt_id", "topic", "factoid"]])
    if g.empty: print(f"WARNING {topic}: no claims"); continue
    ckpt = str(CLUSTER_DIR / f"{slug(topic)}_checkpoint.parquet")
    g = cluster_entailment(nli, g, ckpt, checkpoint_steps=NLI_CHECKPOINT, N=NLI_TOP_N)
    g = break_up_clusters(nli, g, ckpt, checkpoint_steps=NLI_CHECKPOINT, N=NLI_TOP_N)
    unassigned = g.cluster == -1
    g.loc[unassigned, "cluster"] = g.cluster.max() + 1 + np.arange(unassigned.sum())
    g = g.rename(columns={"factoid": "claim"})
    g["class_rep"] = g.groupby("cluster").claim.transform(lambda s: s.mode().iloc[0])
    g.to_parquet(out, index=False)
    print(f"{topic}: {len(g)} claims -> {g.cluster.nunique()} meaning classes "
          f"({unassigned.sum()} unassigned claims made singletons)")

## 4c. Hill–Shannon diversity with coverage-based rarefaction

For each model × topic, the claims form an abundance vector over meaning classes. Diversity is Hill–Shannon diversity, $D_1 = \exp(H)$. Because models produce different numbers of claims, we estimate **sample coverage** (Chao & Jost, 2012) and rarefy every model to the **minimum coverage within each topic**, so models are compared at equally complete samples. This follows Wright et al. using their functions:

- `estimate_coverage`: $\hat C_n = 1 - \frac{f_1}{n}\left[\frac{(n-1)f_1}{(n-1)f_1 + 2f_2}\right]$
- `resample_to_coverage_level`: random subsample whose coverage matches the target. Because this is random, `hsd` is the mean over `N_RESAMPLES` seeded draws (`hsd_sd` is their spread).
- `calculate_diversity`: entropy of the class distribution of the (sub)sample.

As a deterministic robustness check, `hsd_exact` uses the analytic interpolation of Chao et al. (2014, as in iNEXT) (helper functions cell) at the same target coverage.

In [ ]:
import llm_knowledge.epistemic_diversity as ed
from llm_knowledge.epistemic_diversity import resample_to_coverage_level, calculate_diversity

# llm-knowledge's estimate_coverage divides by zero when a sample has no singletons and no doubletons
# (or < 2 claims). Patch it in the module, because resample_to_coverage_level calls it internally.
_estimate_coverage = ed.estimate_coverage
def coverage(d, **kwargs):
    if len(d) < 2: return 0.0
    if (d.cluster.value_counts() == 1).sum() == 0: return 1.0   # no singletons -> complete sample
    return _estimate_coverage(d)
ed.estimate_coverage = coverage

clusters = pd.concat([pd.read_parquet(p) for p in sorted(CLUSTER_DIR.glob("*.parquet"))
                      if not p.stem.endswith("_checkpoint")], ignore_index=True)
rows = []
for topic, g in clusters.groupby("topic"):
    by_cond = dict(tuple(g.groupby("cond")))
    cov = {c: coverage(d) for c, d in by_cond.items()}
    missing = set(CONDITIONS) - {c for c, d in by_cond.items() if len(d) >= 2}
    if missing: print(f"WARNING {topic}: too few claims for {sorted(missing)} (left out)")
    by_cond = {c: d for c, d in by_cond.items() if c not in missing}
    target = min(cov[c] for c in by_cond)
    for c, d in by_cond.items():
        hsd, m = [], []
        for r in range(N_RESAMPLES):
            random.seed(r)
            s = resample_to_coverage_level(d, coverage_level=target)
            hsd.append(np.exp(calculate_diversity(g, sampled_data=s)[0])); m.append(len(s))
        x = d.cluster.value_counts().values
        size, seed = parse_cond(c)
        rows.append({"topic": topic, "cond": c, "size": size, "seed": seed, "size_label": size_label(size),
                     "n_claims": len(d), "n_classes": len(x), "coverage": cov[c], "target_coverage": target,
                     "m": np.mean(m), "hsd": np.mean(hsd), "hsd_sd": np.std(hsd),
                     "hsd_observed": np.exp(calculate_diversity(g, sampled_data=d)[0]),
                     "hsd_exact": exact_rarefied_hsd(x, target)})
div = pd.DataFrame(rows)
div["log_hsd"] = np.log(div.hsd)

# Per-response descriptors, used later as confound checks
desc = gen.groupby(["cond", "topic"]).n_words.mean().rename("mean_words").reset_index()
desc = desc.merge(cl.groupby(["cond", "topic"]).n_claims.mean().rename("claims_per_response").reset_index())
div = div.merge(desc, on=["cond", "topic"], how="left")
div.to_csv(RESULT_DIR / "diversity_per_model_topic.csv", index=False)

print("Target coverage per topic:")
print(div.groupby("topic").target_coverage.first().round(3).to_string())
print("Correlation of resampled vs exact diversity:", round(div[["hsd", "hsd_exact"]].corr().iloc[0, 1], 3))
div.head()

## 5. Analysis

- **Model 1 (main test, fine-tuned models only):** diversity ~ log₂(data size), with crossed random intercepts for topic and seed. A negative slope supports H1.
- **Model 2 (base as reference):** diversity ~ data size as a category with the untuned base as the reference level and a random intercept for topic. Seed is left out of this model because the base model has no seed.
- **Model 3 (robustness):** Model 1 with size as a category, to check whether the change is linear in log size or flattens out (H2 predicts flattening as the model approaches the data's diversity).

In [ ]:
import statsmodels.formula.api as smf
import warnings
Y = DIVERSITY_RESPONSE
levels = ["base"] + [size_label(n) for n in DATA_SIZES]
div["size_cat"] = pd.Categorical(div.size_label, categories=levels)

summary = (div.groupby("size_cat", observed=True)[[Y, "hsd_observed", "n_claims", "claims_per_response", "mean_words"]]
              .agg(["mean", "sem"]).round(3))
summary.to_csv(RESULT_DIR / "summary_by_size.csv")
display(summary)

ft = div[div.cond != "base"].copy()
ft["log2_size_k"] = np.log2(ft["size"] / 1000)
ft["one"] = 1

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m1 = smf.mixedlm(f"{Y} ~ log2_size_k", ft, groups="one", re_formula="0",
                     vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)
    m2 = smf.mixedlm(f"{Y} ~ C(size_cat, Treatment('base'))", div, groups="topic").fit(reml=True)
    ft["size_cat"] = ft.size_cat.cat.remove_unused_categories()
    m3 = smf.mixedlm(f"{Y} ~ C(size_cat)", ft, groups="one", re_formula="0",
                     vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)

for name, m in [("Model 1: log2 size (fine-tuned only)", m1),
                ("Model 2: size category vs. base", m2),
                ("Model 3: size category (fine-tuned only)", m3)]:
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}"); print(m.summary())
    (RESULT_DIR / f"{name.split(':')[0].replace(' ', '_').lower()}.txt").write_text(str(m.summary()))

In [ ]:
# ---- Reading the results against H1 and H2 ----
slope, p_slope = m1.params["log2_size_k"], m1.pvalues["log2_size_k"]
ci = m1.conf_int().loc["log2_size_k"].values
means = div.groupby("size_cat", observed=True)[Y].mean()
print(f"Slope per doubling of data: {slope:+.3f}  (95% CI {ci[0]:+.3f} to {ci[1]:+.3f}, p = {p_slope:.4f})")
print("Mean diversity by condition:", means.round(3).to_dict())

ft_means = means.drop("base")
steps = np.diff(ft_means.values)
print("Change base -> smallest size:", round(ft_means.iloc[0] - means["base"], 3))
print("Changes between consecutive sizes:", np.round(steps, 3))

if p_slope < 0.05 and slope < 0:
    print("\n-> Diversity DECREASES with more data: consistent with H1 (collapse).")
elif p_slope < 0.05 and slope > 0:
    print("\n-> Diversity INCREASES with more data: inconsistent with H1; consistent with H2 "
          "only if the base model starts below Dolly's diversity.")
else:
    print("\n-> No reliable linear trend in log size. Check Model 3 and the base contrast (Model 2): "
          "a jump from base followed by a plateau fits calibration (H2) better than steady collapse (H1).")
print("H2 caveat: Dolly's per-prompt diversity is not measured, so the direction check is suggestive, not conclusive.")

# Per-topic direction of change (how consistent is the effect?)
per_topic = div.pivot_table(index="topic", columns="size_cat", values=Y, aggfunc="mean", observed=True)
per_topic["direction_base_to_max"] = np.sign(per_topic[levels[-1]] - per_topic["base"])
per_topic.round(3)

In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(levels))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

ax = axes[0]
for topic, row in per_topic[levels].iterrows():
    ax.plot(x, row.values, color="0.75", lw=1)
mean, sem = div.groupby("size_cat", observed=True)[Y].mean(), div.groupby("size_cat", observed=True)[Y].sem()
ax.errorbar(x, mean.values, yerr=1.96 * sem.values, color="C0", lw=2.5, marker="o", capsize=4, label="mean ± 95% CI")
ax.set_xticks(x, levels); ax.set_xlabel("fine-tuning data"); ax.set_ylabel(f"Hill–Shannon diversity ({Y})")
ax.set_title("Epistemic diversity (rarefied)"); ax.legend()

ax = axes[1]
for i, col in enumerate(["claims_per_response", "mean_words"]):
    v = div.groupby("size_cat", observed=True)[col].mean()
    ax.plot(x, v.values / v.values.max(), marker="o", label=col)
ax.set_xticks(x, levels); ax.set_title("Possible confounds (scaled to max)"); ax.legend()

ax = axes[2]
for c in [c for c in CONDITIONS if c != "base"]:
    meta = ADAPTER_DIR / c / "train_meta.json"
    if meta.exists():
        h = [r for r in json.loads(meta.read_text())["log_history"] if "loss" in r]
        ax.plot([r["step"] for r in h], [r["loss"] for r in h], lw=1,
                color={DATA_SIZES[0]: "C3", DATA_SIZES[1]: "C1", DATA_SIZES[-1]: "C2"}.get(parse_cond(c)[0], "C0"),
                label=size_label(parse_cond(c)[0]))
h, l = ax.get_legend_handles_labels(); u = dict(zip(l, h))
ax.legend(u.values(), u.keys()); ax.set_xlabel("step"); ax.set_ylabel("train loss"); ax.set_title("Training loss")

plt.tight_layout(); plt.savefig(FIG_DIR / "main_results.png", dpi=200); plt.show()

## 6. Diagnostics and validation

- **Confounds.** If fine-tuned models simply write shorter answers with fewer claims, diversity could drop for that reason alone. Rarefaction to equal coverage handles sample size, but the check below also adds claims per response as a covariate.
- **Overfitting at 1k.** With fixed steps, the 1k subsets are seen many more times. A training loss that keeps falling far below the 10k runs signals memorisation, which is part of what the experiment measures but should be reported.
- **Manual validation.** Export samples for the team to label: is each decomposition faithful and atomic, and do merged claims really mean the same thing? Report agreement in the paper.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    m1c = smf.mixedlm(f"{Y} ~ log2_size_k + claims_per_response", ft, groups="one", re_formula="0",
                      vc_formula={"topic": "0 + C(topic)", "seed": "0 + C(seed)"}).fit(reml=True)
print(m1c.summary())
print("Correlation of diversity with claims per response:",
      round(div[[Y, "claims_per_response"]].corr().iloc[0, 1], 3))

In [ ]:
# 1) Decomposition check: 30 responses with their claims
sample = cl[cl.n_claims > 0].sample(min(30, (cl.n_claims > 0).sum()), random_state=0).merge(
    gen[["cond", "prompt_id", "response"]], on=["cond", "prompt_id"])
sample["claims"] = sample.claims.apply(lambda c: "\n".join(f"- {x}" for x in c))
sample[["cond", "topic", "response", "claims"]].assign(faithful_1to5="", atomic_1to5="").to_csv(
    RESULT_DIR / "validate_decomposition.csv", index=False)

# 2) Clustering check: 50 claims paired with their class representative (merged pairs only)
merged = clusters[clusters.claim != clusters.class_rep]
merged.sample(min(50, len(merged)), random_state=0)[["topic", "claim", "class_rep"]].assign(
    same_meaning_yes_no="").to_csv(RESULT_DIR / "validate_clustering.csv", index=False)

# 3) Largest meaning classes for one topic
t0 = TOPICS[0]
top = clusters[clusters.topic == t0].groupby("class_rep").size().sort_values(ascending=False).head(10)
print(f"Most frequent meaning classes for '{t0}':"); print(top.to_string())
print("\nSaved validation sheets to", RESULT_DIR)

## Notes for the Limitations section

- **Generation precision.** With `GEN_BACKEND="vllm"`, adapters trained on a 4-bit base are applied to a bf16 base at generation time. This is standard practice and applies equally to every model, but `GEN_BACKEND="hf"` matches training exactly if you want to rule it out.
- **Fixed steps.** Holding steps fixed separates data size from training length, but means the 1k models see each example ~10 times. Report the epochs per size and the loss curves.
- **Base-model outputs.** The untuned base often continues the text instead of answering. Report the share of empty or claim-free responses per model (Stage 3 and 4a tables), since this affects the base reference point.
- **Pipeline choices.** Decomposition prompt, clustering, and diversity use Wright et al.'s own `llm-knowledge` package (v0.1.3). Differences from their setup: the 70B decomposition model is called through an OpenAI-compatible endpoint instead of in-process vLLM, and claims left unassigned after `break_up_clusters` are counted as singleton classes. Useful robustness checks: re-run 4b–5 with `NLI_TOP_N` at 3 and 10, and with `DIVERSITY_RESPONSE = "hsd_exact"`.
- **Random rarefaction.** `resample_to_coverage_level` draws a random subsample; `hsd` averages `N_RESAMPLES` seeded draws and `hsd_sd` reports their spread.
- **Small design.** 10 topics × 3 seeds gives limited power; random-effect variances (especially for seed) may be estimated near zero, which statsmodels reports as a convergence warning rather than an error.
- **H2.** Dolly's own per-prompt diversity is not measured, so the H2 direction check is suggestive only.